# RAG — Stage 2: Indexing (embeddings + FAISS)

**Goal:** map each text chunk to a **dense vector** (OpenAI embeddings) and store them in a **FAISS** index for fast similarity search.

**Requirements:** `OPENAI_API_KEY` in `rag_lab/.env` (embeddings are billed).

**What you will see:** chunk count, embedding model name, and confirmation that the index was built in memory.

**Next notebook:** Stage 3 runs queries against this index (retrieval only, no LLM answer yet).


In [1]:
%pip install -q langchain-text-splitters langchain-core langchain-community langchain-openai faiss-cpu pymupdf openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

# Run this notebook from the notebooks/ folder (parent = rag_lab).
RAG_LAB = Path("..").resolve()
SRC = RAG_LAB / "src"
sys.path.insert(0, str(SRC))

from qbrain_rag.application.chunking import chunk_text
from qbrain_rag.config.notebook_defaults import resolve_default_srs_path
from qbrain_rag.config.settings import get_settings
from qbrain_rag.infrastructure.document_loaders import load_document
from qbrain_rag.infrastructure.vector_store import build_faiss_store

srs_path = resolve_default_srs_path(RAG_LAB)
text = load_document(str(srs_path))
chunks = chunk_text(text)
metadata = [{"source_file": srs_path.name, "chunk_id": i + 1} for i in range(len(chunks))]

s = get_settings()
print("Embedding model:", s.embedding_model)

store = build_faiss_store(chunks, metadata)
print("Indexed", len(chunks), "chunks into FAISS (in-memory).")
print("Store type:", type(store).__name__)


Embedding model: text-embedding-3-small


Indexed 30 chunks into FAISS (in-memory).
Store type: FAISS


### Verify: vectors **and** chunk text

FAISS stores **embedding vectors**; LangChain also keeps each chunk’s full text in an **in-memory docstore** (so retrieval returns `Document.page_content`, not only an id).

Run the cell below to assert every chunk string is present in the store.

In [3]:
from qbrain_rag.infrastructure.vector_store import (
    chunk_texts_materialized_in_store,
    indexed_document_count,
)

assert indexed_document_count(store) == len(chunks)
assert chunk_texts_materialized_in_store(store, chunks)
print("OK: docstore contains all", len(chunks), "chunk texts (matches FAISS vector count).")

OK: docstore contains all 30 chunk texts (matches FAISS vector count).
